In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Decision Trees
Despite their age, tree-based models are still being used in finance because:
- Models like Random Forests and Gradient Boosted Trees (XGBoost, LightGBM) output feature importance metrics, which help quants identify which market signals actually drive returns.
- Financial data is full of extreme events. Tree splits are based on rank order rather than exact distances, making them far less sensitive to outliers than linear regression or neural networks.

Decision Trees work by pulling a dataset's features and making sequencial decisions at decision nodes based off of them. It has a similar structure to 20 questions where the model goes through the features and asks a bunch of branching yes and no questions (leaf nodes) before making a prediction. 

## Splitting
But how do we determine a yes or no and better yet how do we even ask the question? This is asking how we split our data using the leaf nodes. The best split is basically a split where after spliting each split has data points where every data point has the same class. There are a few ways to do this, Gini impurity is a common one: $$\text{Gini}=1-\sum^{n}_{i=1}p^2_i$$
Where:
- $p_i$ is the probability/proportion of a specific class
- $n$ is the total number of classes

The model tests different ways to split the data and then choses the one that reduces the overall impurity the most (also called information gain).

You can also use entropy defined as: $$H(S)=-\sum^{n}_{i=1}p_i\log_2(p_i)$$
Where:
- $S$ is the current dataset/node
- $p_i$ is the probability/proportion of a specific class
- $n$ is the total number of classes

Metric similar to Gini, low entropy means data mostly belongs to a single class, high entropy means data is very mixed class-wise.

Reasons to use Gini over entropy:
- Large datasets and deep trees
- If you need fast training speed
- Calculation time is faster than entropy because of $O(n)$ complexity
- Behaviour: Favors creating partitions that isolate the largest, most dominant class

Reasons to use entropy over Gini:
- Smaller datasets
- Has $nO(\log_2(n))$ complexity
- Behaviour: Tends to produce slightly broader, well-balanced tree branches

## Information Gain
Information Gain ($IG$) measures the reduction in entropy or Gini impurity after a dataset is split on an attribute. The model calculates the impurity of the parent node and subtracts the weighted impurity of the child nodes. The split that maximizes this difference is selected. Mathematically, for a parent node $D$ split into $k$ children partitions:$$IG(D, A) = I(D) - \sum_{j=1}^{k} \frac{|D_j|}{|D|} I(D_j)$$Where:
- $I(\cdot)$ is the impurity measure (either Entropy or Gini)
- $|D|$ is the total number of samples in the parent node
- $|D_j|$ is the number of samples in the $j$-th child node.

Standard Information Gain naturally biases toward features with a large number of distinct values (e.g., a high-frequency timestamp variable). To counteract this, algorithms like C4.5 use Gain Ratio, which penalizes features with high intrinsic information (features that split the data into many small, unhelpful partitions).

# Overfitting and Regularization (Pruning)
A primary weakness of single decision trees is their high variance. If left unchecked, a tree will grow deep enough to perfectly memorize the noise in financial training data. Here are some common precdeures to prevent this:
## 1. Pre-Pruning (Hyperparameters)

Stopping the tree from growing before it perfectly fits the data. Constraints include
- max_depth
- min_samples

Where max_depth controls the maximum number of levels a tree can grow during training and min_samples sets the minimum number of data points required for an internal node to be divided further.

## 2. Post-Pruning

Allowing the tree to grow completely, and then collapsing subtrees that do not provide significant predictive power. This is done by minimizing a cost function that penalizes tree complexity:$$R_\alpha(T) = R(T) + \alpha |T|$$Where:
- $R(T)$ is the total misclassification rate (or total MSE) of the tree
- $|T|$ is the number of terminal leaves in the tree
- $\alpha$ is the complexity parameter (tuned via cross-validation). A higher $\alpha$ forces a smaller, simpler tree.

In [3]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, gain=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.gain = gain
        self.value = value

class DT:
    def __init__(self, max_depth=4, min_samples=2):
        self.max_depth = max_depth
        self.min_samples = min_samples
        
    def entropy(self, y):
        if len(y) == 0:
            return 0

        entropy = 0
        labels = np.unique(y)

        for label in labels:
            p = len(y[y == label]) / len(y)
            entropy += -p * np.log2(p)

        return entropy

    def info_gain(self, parent, left, right):
        parent_e = self.entropy(parent)

        w_l = len(left) / len(parent)
        w_r = len(right) / len(parent)

        e_l = self.entropy(left)
        e_r = self.entropy(right)

        weighted_e = (w_l * e_l) + (w_r * e_r)

        return parent_e - weighted_e

    def best_split(self, dataset, num_features):   # These "features" represent the independant variables you are feeding in i.e. the columns of your dataset
        best_split = {}
        max_gain = -float("inf")

        for feature_idx in range(num_features):
            feature_values = dataset[:, feature_idx]
            thresholds = np.unique(feature_values)

            for threshold in thresholds:
                left = dataset[dataset[:, feature_idx] <= threshold]
                right = dataset[dataset[:, feature_idx] > threshold]

                if len(left) == 0 or len(right) == 0:
                    continue

                y = dataset[:, -1]
                y_left = left[:, -1]
                y_right = right[:, -1]

                gain = self.info_gain(y, y_left, y_right)

                if gain > max_gain:
                    best_split = {
                        "feature": feature_idx,
                        "threshold": threshold,
                        "left": left,
                        "right": right,
                        "gain": gain
                    }
                    max_gain = gain

        return best_split

    def build_tree(self, dataset, depth=0):
        X = dataset[:, :-1]
        y = dataset[:, -1]
        n_samples, n_features = X.shape

        if (
            n_samples >= self.min_samples
            and depth <= self.max_depth
        ):
            split = self.best_split(dataset, n_features)

            if split and split["gain"] > 0:
                left_subtree = self.build_tree(split["left"], depth + 1)
                right_subtree = self.build_tree(split["right"], depth + 1)

                return Node(
                    feature=split["feature"],
                    threshold=split["threshold"],
                    left=left_subtree,
                    right=right_subtree,
                    gain=split["gain"]
                )

        # Leaf node
        leaf_value = self._majority_vote(y)
        return Node(value=leaf_value)

    def _majority_vote(self, y):
        return max(set(y), key=list(y).count)
    
    def fit(self, X, y):
        dataset = np.concatenate((X, y.reshape(-1, 1)), axis=1)
        self.root = self.build_tree(dataset)

    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

In [5]:
data = load_iris()
X, y = data.data[:100], data.target[:100] 
    
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DT(max_depth=3, min_samples=2)
dt.fit(X_train, y_train)
    
preds = dt.predict(X_test)
    
    # Calculate accuracy
accuracy = np.mean(preds == y_test)
print(f"Iris Dataset Accuracy: {accuracy * 100:.2f}%")

Iris Dataset Accuracy: 100.00%


Damn, works better than I thought

# Random Forests

If Decision Trees are so great then why would they ever make Random Forests?
- Random Forests are a lot better at dealing with overfitting. A single decision tree can grow deep enough to memorize training data perfectly, including its noise. Random forests combat this by averaging out predictions, ensuring the model generalizes well to unseen data
- Decision Trees have very high variance, at times small changes to training data can vastly impact how a tree splits. But since a Random Forest pools from multiple trees it does not have this problem.
- For finance the feature importance thing can be very important to see which metrics are most helpful

To make sure the trees don't end up the same, there needs to be an aspect of randomness added. There are two ways of doing this.
1. Bootstrapping (Row Sampling)

The model will select a random subset of samples to train on (with replacement)

2. Random Subspace Method (Column Sampling)

At each node of every tree, the algorithm chooses a random subset of features. This forces trees to discover alternative rules and prevents dominant features from dictating every single tree.
- For classification tasks: $\sqrt{M}$ (where $M$ is the total number of features)
- For regression tasks: $\frac{M}{3}$

Predictions work by a majority vote, if most trees think something is one class then it is ruled that way.

In [13]:
class DT:
    def __init__(self, max_depth=4, min_samples=2):
        self.max_depth = max_depth
        self.min_samples = min_samples
        
    def entropy(self, y):
        if len(y) == 0:
            return 0

        entropy = 0
        labels = np.unique(y)

        for label in labels:
            p = len(y[y == label]) / len(y)
            entropy += -p * np.log2(p)

        return entropy

    def info_gain(self, parent, left, right):
        parent_e = self.entropy(parent)

        w_l = len(left) / len(parent)
        w_r = len(right) / len(parent)

        e_l = self.entropy(left)
        e_r = self.entropy(right)

        weighted_e = (w_l * e_l) + (w_r * e_r)

        return parent_e - weighted_e

    def best_split(self, dataset, num_features):   # These "features" represent the independant variables you are feeding in i.e. the columns of your dataset
        best_split = {}
        max_gain = -float("inf")

        for feature_idx in range(num_features):
            feature_values = dataset[:, feature_idx]
            thresholds = np.unique(feature_values)

            for threshold in thresholds:
                left = dataset[dataset[:, feature_idx] <= threshold]
                right = dataset[dataset[:, feature_idx] > threshold]

                if len(left) == 0 or len(right) == 0:
                    continue

                y = dataset[:, -1]
                y_left = left[:, -1]
                y_right = right[:, -1]

                gain = self.info_gain(y, y_left, y_right)

                if gain > max_gain:
                    best_split = {
                        "feature": feature_idx,
                        "threshold": threshold,
                        "left": left,
                        "right": right,
                        "gain": gain
                    }
                    max_gain = gain

        return best_split

    def build_tree(self, dataset, depth=0):
        X = dataset[:, :-1]
        y = dataset[:, -1]
        n_samples, n_features = X.shape

        if (
            n_samples >= self.min_samples
            and depth <= self.max_depth
        ):
            split = self.best_split(dataset, n_features)

            if split and split["gain"] > 0:
                left_subtree = self.build_tree(split["left"], depth + 1)
                right_subtree = self.build_tree(split["right"], depth + 1)

                return Node(
                    feature=split["feature"],
                    threshold=split["threshold"],
                    left=left_subtree,
                    right=right_subtree,
                    gain=split["gain"]
                )

        # Leaf node
        leaf_value = self._majority_vote(y)
        return Node(value=leaf_value)

    # Needed to add this function
    def get_feature_importance(self, num_features):
        importances = np.zeros(num_features)
    
        def _collect_importance(node):
            if node is None or node.feature is None:
                return
        
            importances[node.feature] += node.gain
        
            _collect_importance(node.left)
            _collect_importance(node.right)

        _collect_importance(self.root)
        return importances

    def _majority_vote(self, y):
        return max(set(y), key=list(y).count)
    
    def fit(self, X, y):
        dataset = np.concatenate((X, y.reshape(-1, 1)), axis=1)
        self.root = self.build_tree(dataset)

    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

class RForest:
    def __init__(self, num_trees=10, max_depth=10, min_samples=2, max_features=None):
        self.num_trees = num_trees
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.max_features = max_features
        self.trees = []

    def _bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]

    def get_feature_importances(self, X):
        n_features = X.shape[1]
        total_importances = np.zeros(n_features)

        for tree in self.trees:
            total_importances += tree.get_feature_importance(n_features)

        if np.sum(total_importances) > 0:
            total_importances /= np.sum(total_importances)

        return total_importances

    def fit(self, X, y):
        self.trees = []

        for _ in range(self.num_trees):
            tree = DT(
                min_samples=self.min_samples,
                max_depth=self.max_depth
            )

            X_sample, y_sample = self._bootstrap_sample(X, y)

            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])

        # majority vote across trees
        return np.array([
            max(set(row), key=list(row).count)
            for row in tree_preds.T
        ])

In [14]:
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
print("Feature names: ", feature_names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

rf = RForest(num_trees=15, max_depth=5, min_samples=2)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Test Accuracy: {accuracy * 100:.2f}%")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

importances = rf.get_feature_importances(X_train)
    
# Sort features by their information gain score
indices = np.argsort(importances)[::-1]
    
print("Top 5 features:")
for i in range(min(5, len(indices))):
    idx = indices[i]
    print(f"{i+1}. Feature: {feature_names[idx]:<20} | Importance Score: {importances[idx]:.4f}")

Feature names:  ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']
Overall Test Accuracy: 96.30%
Classification Report:
              precision    recall  f1-score   support

     class_0       0.90      1.00      0.95        18
     class_1       1.00      0.90      0.95        21
     class_2       1.00      1.00      1.00        15

    accuracy                           0.96        54
   macro avg       0.97      0.97      0.97        54
weighted avg       0.97      0.96      0.96        54

Top 5 features:
1. Feature: alcohol              | Importance Score: 0.2912
2. Feature: flavanoids           | Importance Score: 0.2329
3. Feature: color_intensity      | Importance Score: 0.1830
4. Feature: od280/od315_of_diluted_wines | Importance Score: 0.0772
5. Feature: proline              | Importance Score: 0.0654
